In [1]:
!pip install -q \
  transformers \
  huggingface_hub \
  evaluate

!pip install -U bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.4/59.4 MB 12.9 MB/s eta 0:00:00


In [2]:
import re
import torch
from peft import PeftModel
from dataclasses import dataclass
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from huggingface_hub import hf_hub_download, login

In [3]:
login()

In [4]:
# -----------------------------
# 0) Helpers / configuration
# -----------------------------
LABELS = {"negative", "neutral", "positive"}
SPLIT = re.compile(r'(?<=[.!?])\s+')

def label_to_score(lbl: str) -> float:
    lbl = lbl.strip().lower()
    if lbl == "positive": return +1.0
    if lbl == "neutral":  return  0.0
    if lbl == "negative": return -1.0
    # fallback if model outputs unexpected token
    return 0.0

In [15]:
# -----------------------------
# 1) Single-turn classification
#    (discrete label; your TinyLLaMA generate)
# -----------------------------
def classify_label(model, tokenizer, text: str, device: str) -> str:
    instruction = (
        "### Instruction:\n"
        "Klassifiziere die Stimmung der folgenden Bewertung als 'positive', 'neutral' oder 'negative'.\n\n"
        "### Bewertung:\n"
    )
    answer_prefix = "\n\n### Antwort:\n"
    prompt = instruction + text + answer_prefix

    # fast & safe defaults
    tokenizer.truncation_side = "left"
    inputs = tokenizer(prompt, return_tensors="pt",
                       truncation=True, max_length=2046, padding=False).to(device)

    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=2,          # we only expect one label token
            do_sample=False,
            use_cache=False,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id
        )

    decoded = tokenizer.decode(out[0], skip_special_tokens=True)
    # robust extraction
    answer_part = decoded.split("### Antwort:")[-1] if "### Antwort:" in decoded else decoded
    label = (answer_part.strip().split() or [""])[0].lower()
    return label if label in LABELS else "neutral"

In [22]:
# -----------------------------
# 2) Message-level sentiment
#    (multi-sentence → length-weighted average of label scores)
# -----------------------------
def sentiment_score(model, tokenizer, device, text: str, max_sents: int = 12) -> float:
    sents = [s.strip() for s in SPLIT.split(text) if s.strip()]
    if not sents:
        return 0.0
    # cap for speed
    sents = sents[:max_sents]

    scores, lengths = [], []
    for s in sents:
        lbl = classify_label(model, tokenizer, s, device)
        scores.append(label_to_score(lbl))
        lengths.append(len(s))

    w = sum(lengths) or 1
    return sum(sc * L for sc, L in zip(scores, lengths)) / w  # [-1..1]

In [28]:
# -----------------------------
# 3) Smoothing + trend reversal
# -----------------------------
class EMA:
    def __init__(self, alpha=0.35):
        self.a, self.v = alpha, None
    def update(self, x: float) -> float:
        self.v = x if self.v is None else (self.a * x + (1 - self.a) * self.v)
        return self.v

class Trend:
    def __init__(self, up_thr=+0.06, down_thr=-0.06, sustain=2):
        self.up_thr = up_thr
        self.down_thr = down_thr
        self.sustain = sustain

        self.state = "flat"       # committed trend: "up" | "down" | "flat"
        self.prev = None          # last s_ema
        self.pending_dir = None   # last non-flat desired direction we are accumulating
        self.count = 0            # consecutive steps toward pending_dir

    def update(self, s_ema):
        if self.prev is None:
            self.prev = s_ema
            # first value: nothing to compare against
            return self.state, None, "flat"

        delta = s_ema - self.prev
        self.prev = s_ema

        # decide instantaneous desired direction from delta
        if   delta >= self.up_thr:   desired = "up"
        elif delta <= self.down_thr: desired = "down"
        else:                         desired = "flat"

        event = None

        if desired == "flat":
            # no clear movement → reset pending evidence and (soft) go flat
            self.pending_dir = None
            self.count = 0
            self.state = "flat"

        elif desired == self.state:
            # already committed to this direction → nothing to prove
            self.pending_dir = None
            self.count = 0

        else:
            # desired is a directional change away from current state
            # accumulate ONLY if the direction matches the current pending_dir
            if desired == self.pending_dir:
                self.count += 1
            else:
                self.pending_dir = desired
                self.count = 1

            if self.count >= self.sustain:
                self.state = desired
                self.pending_dir = None
                self.count = 0
                event = f"reversal_to_{desired}"

        return self.state, event, desired

In [29]:
# -----------------------------
# 4) Orchestrator
# -----------------------------
@dataclass
class TurnInfo:
    who: str           # "human" | "bot"
    text: str
    s_raw: float       # per-turn score in [-1..1]
    s_ema: float       # smoothed
    trend: str         # "up"|"down"|"flat"
    event: str | None  # "reversal_to_up"/"reversal_to_down"/None
    desired: str       # "up"|"down"|"flat" (instant direction before sustain)

class SentimentTracker:
    def __init__(self, model, tokenizer, device,
                 alpha=0.35, up_thr=0.06, down_thr=-0.06, sustain=2):
        self.model, self.tokenizer, self.device = model, tokenizer, device
        self.ema = EMA(alpha)
        self.trend = Trend(up_thr, down_thr, sustain)
        self.history: list[TurnInfo] = []

    def step(self, who: str, text: str) -> TurnInfo:
        raw = sentiment_score(self.model, self.tokenizer, self.device, text)
        sm  = self.ema.update(raw)
        tr, ev, desired = self.trend.update(sm)
        info = TurnInfo(who, text, raw, sm, tr, ev, desired)
        self.history.append(info)
        return info

In [9]:
device = "cuda" # Used in colab

base_model_id = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
lora_repo_id  = "eduhuemar001/tinyllama-german-sentiment-4bit-v7"
subfolder = "adapters/epoch_005"
tokenizer = AutoTokenizer.from_pretrained(base_model_id)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/551 [00:00<?, ?B/s]

In [10]:
# 4-bit quantization config
compute_dtype = torch.bfloat16 if (torch.cuda.is_available() and torch.cuda.get_device_capability(0)[0] >= 8) else torch.float16
quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=compute_dtype,
)

# Load base model in 4-bit
base_model = AutoModelForCausalLM.from_pretrained(
    base_model_id,
    quantization_config=quant_config,
    device_map="auto",
)

# Attach LoRA adapters
lora_model = PeftModel.from_pretrained(base_model, lora_repo_id, subfolder=subfolder)
#lora_model = PeftModel.from_pretrained(base_model, lora_repo_id)

lora_model = lora_model.to(device)
lora_model.eval()

config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

adapter_config.json:   0%|          | 0.00/902 [00:00<?, ?B/s]

adapters/epoch_005/adapter_model.safeten(…):   0%|          | 0.00/61.3M [00:00<?, ?B/s]

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): LlamaForCausalLM(
      (model): LlamaModel(
        (embed_tokens): Embedding(32000, 2048)
        (layers): ModuleList(
          (0-21): 22 x LlamaDecoderLayer(
            (self_attn): LlamaAttention(
              (q_proj): lora.Linear4bit(
                (base_layer): Linear4bit(in_features=2048, out_features=2048, bias=False)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.058, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=2048, out_features=32, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=32, out_features=2048, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj): Lin

In [31]:
# --- Load Qwen model (4-bit if GPU available) ---
QWEN_ID = "Qwen/Qwen2.5-1.5B-Instruct"
use_4bit = torch.cuda.is_available()
bnb = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16
) if use_4bit else None

q_tok = AutoTokenizer.from_pretrained(QWEN_ID, use_fast=True)
q_mdl = AutoModelForCausalLM.from_pretrained(
    QWEN_ID,
    device_map="auto" if torch.cuda.is_available() else None,
    torch_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
    quantization_config=bnb
)

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

In [32]:
def qwen_reply(history, user_text, max_new_tokens=320):
    msgs = history + [{"role": "user", "content": user_text}]
    prompt = q_tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    x = q_tok(prompt, return_tensors="pt")
    if torch.cuda.is_available():
        x = {k: v.to(q_mdl.device) for k, v in x.items()}
    y = q_mdl.generate(**x, max_new_tokens=max_new_tokens, do_sample=False,
                       eos_token_id=q_tok.eos_token_id, pad_token_id=q_tok.eos_token_id)
    gen = q_tok.decode(y[0][x["input_ids"].shape[-1]:], skip_special_tokens=True).strip()
    return gen

In [30]:
# -----------------------------
# 5) Example
# -----------------------------

tracker = SentimentTracker(lora_model, tokenizer, device, alpha=0.35, up_thr=0.06, down_thr=-0.06, sustain=2)

dialog = [
    # --- NEGATIVE / FRUSTRATED START (strongly negative sentiment) ---
    ("human", "Ich bin wirklich enttäuscht. Seit Tagen habe ich keine Antwort bekommen."),
    ("bot",   "Es tut mir leid, dass Sie warten mussten. Ich sehe mir das sofort an."),
    ("human", "Das ist schon das zweite Mal, dass so etwas passiert."),
    ("bot",   "Das sollte natürlich nicht vorkommen. Ich prüfe gleich, woran es liegt."),
    ("human", "Langsam habe ich das Gefühl, dass meine Anliegen nicht ernst genommen werden."),
    ("bot",   "Ich verstehe Ihren Ärger. Wir möchten das auf keinen Fall so wirken lassen."),
    ("human", "Ich bin ehrlich gesagt kurz davor, den Anbieter zu wechseln."),
    ("bot",   "Ich hoffe, wir können das noch verhindern. Lassen Sie mich kurz nachsehen."),
    ("human", "Ich hoffe, das ist nicht wieder nur eine leere Versprechung."),
    ("bot",   "Nein, ich gebe mein Bestes, Ihnen direkt zu helfen."),

    # --- NEUTRAL / CLARIFYING PHASE (emotion stabilizes) ---
    ("bot",   "Können Sie mir bitte Ihre Kundennummer nennen, damit ich den Fall prüfen kann?"),
    ("human", "Ja, die Nummer ist 34821."),
    ("bot",   "Perfekt, vielen Dank. Einen Moment bitte, ich überprüfe die Daten."),
    ("human", "Kein Problem, ich warte."),
    ("bot",   "Ich sehe, dass Ihr Anliegen noch in Bearbeitung ist."),
    ("human", "Aha, also noch nicht abgeschlossen. Wissen Sie, woran es hängt?"),
    ("bot",   "Laut System gab es eine interne Verzögerung, die ich jetzt korrigieren kann."),
    ("human", "Gut, danke fürs Nachsehen."),
    ("bot",   "Ich leite das sofort an die zuständige Stelle weiter."),
    ("human", "Okay, ich hoffe, das klappt diesmal."),

    # --- POSITIVE RESOLUTION PHASE (positive sentiment) ---
    ("bot",   "Gute Nachricht: Der Vorgang wurde gerade bestätigt."),
    ("human", "Oh, das ist ja super! Vielen Dank."),
    ("bot",   "Ich freue mich, dass es jetzt funktioniert."),
    ("human", "Ich bin echt erleichtert, das war ziemlich nervig."),
    ("bot",   "Verstehe ich gut. Hauptsache, das Problem ist gelöst."),
    ("human", "Ja, absolut. Danke für Ihre Geduld und Hilfe."),
    ("bot",   "Sehr gerne! Ich wünsche Ihnen noch einen angenehmen Tag."),
    ("human", "Ihnen auch, danke nochmals."),
    ("bot",   "Falls künftig etwas sein sollte, melden Sie sich gerne."),
    ("human", "Mache ich, vielen Dank nochmal für die schnelle Lösung."),
]

for i, (who, text) in enumerate(dialog, 1):
    if who != "human":
        continue  # skip bot messages

    info = tracker.step(who, text)
    flag = f"  -> {info.event}" if info.event else ""
    print(
        f"{i:02d} "
        f"[{who:<5}] "
        f"s_raw ={info.s_raw:+7.2f}  "
        f"s_ema ={info.s_ema:+7.2f}  "
        f"desired={info.desired:<6}  "
        f"trend={info.trend:<6}"
        f"{flag}"
    )

01 [human] s_raw =  -1.00  s_ema =  -1.00  desired=flat    trend=flat  
03 [human] s_raw =  -1.00  s_ema =  -1.00  desired=flat    trend=flat  
05 [human] s_raw =  -1.00  s_ema =  -1.00  desired=flat    trend=flat  
07 [human] s_raw =  +0.00  s_ema =  -0.65  desired=up      trend=flat  
09 [human] s_raw =  -1.00  s_ema =  -0.77  desired=down    trend=flat  
12 [human] s_raw =  +0.00  s_ema =  -0.50  desired=up      trend=flat  
14 [human] s_raw =  +1.00  s_ema =  +0.02  desired=up      trend=up      -> reversal_to_up
16 [human] s_raw =  -0.56  s_ema =  -0.18  desired=down    trend=up    
18 [human] s_raw =  +1.00  s_ema =  +0.23  desired=up      trend=up    
20 [human] s_raw =  -1.00  s_ema =  -0.20  desired=down    trend=up    
22 [human] s_raw =  +1.00  s_ema =  +0.22  desired=up      trend=up    
24 [human] s_raw =  -1.00  s_ema =  -0.21  desired=down    trend=up    
26 [human] s_raw =  +1.00  s_ema =  +0.22  desired=up      trend=up    
28 [human] s_raw =  +1.00  s_ema =  +0.49  de

In [ ]:
tracker = SentimentTracker(alpha=0.35, up_thr=0.06, down_thr=-0.06, sustain=2)
history = [{"role": "system", "content": "Du bist ein hilfreicher deutschsprachiger Assistent."}]

print("Chat gestartet. Leere Eingabe = Ende.\n")
turn = 0

while True:
    user = input("Du: ").strip()
    if not user:
        break

    # Analyze only human sentiment
    turn += 1
    info = tracker.step("human", user)
    flag = f"  -> {info.event}" if info.event else ""
    print(
        f"{turn:02d} [human] "
        f"s_raw={info.s_raw:+7.2f}  "
        f"s_ema={info.s_ema:+7.2f}  "
        f"desired={info.desired:<6}  "
        f"trend={info.trend:<6}"
        f"{flag}"
    )

    # Get Qwen response
    reply = qwen_reply(history, user)
    print(f"Qwen: {reply}\n")

    history.append({"role": "user", "content": user})
    history.append({"role": "assistant", "content": reply})